In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import pearsonr

plt.style.use('seaborn-v0_8-whitegrid')
FAST_TRACK = True

In [ ]:
def aic(n, k_params, rss):
    return n * np.log(rss / n + 1e-12) + 2.0 * k_params

def fit_logarithmic(xi, lmin):
    log_xi = np.log(xi)
    A      = np.vstack([log_xi, np.ones(len(xi))]).T
    coef, _, _, _ = np.linalg.lstsq(A, lmin, rcond=None)
    rss = float(((lmin - A @ coef) ** 2).sum())
    return {'k_c': coef[0], 'intercept': coef[1],
            'aic': aic(len(xi), 2, rss), 'y_pred': A @ coef}

def fit_linear(xi, lmin):
    A = np.vstack([xi, np.ones(len(xi))]).T
    coef, _, _, _ = np.linalg.lstsq(A, lmin, rcond=None)
    rss = float(((lmin - A @ coef) ** 2).sum())
    return {'slope': coef[0], 'intercept': coef[1],
            'aic': aic(len(xi), 2, rss), 'y_pred': A @ coef}

def fit_power_law(xi, lmin):
    def model(x, a, nu):
        return a * np.power(np.maximum(x, 1e-6), nu)
    try:
        popt, _ = curve_fit(model, xi, lmin, p0=[1.0, 1.0],
                            bounds=([0.0, 0.0], [np.inf, 5.0]))
        rss = float(((lmin - model(xi, *popt)) ** 2).sum())
        return {'a': popt[0], 'nu': popt[1],
                'aic': aic(len(xi), 2, rss), 'y_pred': model(xi, *popt)}
    except RuntimeError:
        return {'aic': float('inf'), 'y_pred': lmin * 0}

In [ ]:
if FAST_TRACK:
    rng         = np.random.default_rng(42)
    xi_values   = np.array([2.0, 3.5, 5.0, 8.0, 12.0, 20.0, 35.0, 50.0])
    lmin_values = 2.1 * np.log(xi_values) + 2.8 + 0.15 * rng.standard_normal(8)
    print('Mode: FAST-TRACK (synthetic data)')
else:
    import json
    with open('results/h2/h2_results.json') as fh:
        raw = json.load(fh)
    pairs       = [(v['xi'], v['lmin_mean']) for v in raw['records'].values()
                   if not np.isnan(v['lmin_mean'])]
    xi_values   = np.array([p[0] for p in pairs])
    lmin_values = np.array([p[1] for p in pairs])
    print(f'Mode: FULL ({len(xi_values)} data points loaded)')

print(f'xi range:    [{xi_values.min():.2f}, {xi_values.max():.2f}]')
print(f'L_min range: [{lmin_values.min():.2f}, {lmin_values.max():.2f}]')

In [ ]:
log_fit   = fit_logarithmic(xi_values, lmin_values)
lin_fit   = fit_linear(xi_values, lmin_values)
power_fit = fit_power_law(xi_values, lmin_values)

models   = {'Logarithmic': log_fit, 'Linear': lin_fit, 'Power-law': power_fit}
best     = min(models, key=lambda m: models[m]['aic'])
best_aic = models[best]['aic']
r, p     = pearsonr(np.log(xi_values), lmin_values)

print(f'{"Model":<14} {"AIC":>10} {"DELTA_AIC":>10}')
print('-' * 36)
for name, res in models.items():
    print(f'{name:<14} {res["aic"]:>10.3f} {res["aic"] - best_aic:>10.3f}')

print(f'\nBest model:  {best}')
print(f'Fitted k_c:  {log_fit["k_c"]:.4f}')
print(f'Pearson r:   {r:.4f}  (p = {p:.4f})')

In [ ]:
xi_dense     = np.linspace(xi_values.min() * 0.9, xi_values.max() * 1.1, 300)
log_xi_dense = np.log(xi_dense)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(xi_values, lmin_values, color='#2166ac', zorder=5, s=60)
axes[0].plot(xi_dense,
             log_fit['k_c'] * log_xi_dense + log_fit['intercept'],
             color='#1a9641', linewidth=2,
             label=f'Log fit  k_c={log_fit["k_c"]:.3f}')
axes[0].plot(xi_dense,
             lin_fit['slope'] * xi_dense + lin_fit['intercept'],
             color='#d6604d', linewidth=1.5, linestyle='--', label='Linear fit')
axes[0].set_xlabel('xi')
axes[0].set_ylabel('L_min')
axes[0].set_title('H2: Model Comparison')
axes[0].legend()

axes[1].scatter(np.log(xi_values), lmin_values, color='#2166ac', zorder=5, s=60)
axes[1].plot(log_xi_dense,
             log_fit['k_c'] * log_xi_dense + log_fit['intercept'],
             color='#1a9641', linewidth=2)
axes[1].set_xlabel('log(xi)')
axes[1].set_ylabel('L_min')
axes[1].set_title(f'H2: Log-Space Linearity  (r = {r:.4f})')

plt.tight_layout()
plt.savefig('nb03_h2_depth_scaling.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
h2_validated = (best == 'Logarithmic') and (abs(r) > 0.90) and (p < 0.05)
print(f'Log model wins AIC:  {best == "Logarithmic"}')
print(f'|r| > 0.90:          {abs(r) > 0.90}  ({abs(r):.4f})')
print(f'p < 0.05:            {p < 0.05}  ({p:.4f})')
print(f'H2 VALIDATED:        {h2_validated}')